In [1]:
import numpy as np

from numba import njit

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

import plotly.graph_objects as go

import fastplotlib as fpl

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),Apple M4,IntegratedGPU,Metal,


To silence this warning, use a fully namespaced name.


In [2]:
np.random.seed(42)

# Spring Chain

In [3]:
time = 1000
dt = 0.005
steps = int(time / dt)
tau_steps = int(1.5 / dt)

test_size = 0.2

N = 10

m_diag = np.random.uniform(0.1, 0.5, size=N)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(np.random.uniform(0.1, 0.3, size=N))

k_vals = np.random.uniform(0.5, 8, N - 1)
K = np.zeros((N, N))
idx = np.arange(N - 1)
K[idx, idx + 1] = -k_vals
K[idx + 1, idx] = -k_vals
k_padded = np.pad(k_vals, (1, 1), mode="constant")
K[np.arange(N), np.arange(N)] = k_padded[:-1] + k_padded[1:]


t = np.linspace(0, time, steps)

u = np.sin(t)

In [4]:
@njit
def run_simulation(steps, dt, u, M_INV, K, C, N):
    x = np.zeros((steps, N))
    v = np.zeros((steps, N))

    acc = np.zeros(N)

    for i in range(1, steps):
        u_force = np.zeros(N)
        u_force[0] = u[i - 1]

        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + u_force)

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        u_force_next = np.zeros(N)
        u_force_next[0] = u[i]

        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + u_force_next)

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [5]:
x, v = run_simulation(steps, dt, u, M_INV, K, DAMP, N)

In [6]:
x_init = np.arange(N).astype(np.float32)
y_init = np.zeros(N).astype(np.float32)

dots_data = np.column_stack([x_init, y_init])


fig = fpl.Figure()
dots_graphic = fig[0, 0].add_scatter(data=dots_data, sizes=15, colors="magenta")

frame_tracker = 0
frames = 5
max_frames = 2000
def update_springs(canvas):
    global frame_tracker

    frame_tracker = (frame_tracker + frames) % steps
    print(f"Frame: {frame_tracker}/{steps}", end="\r")

    if frame_tracker >= max_frames:
        canvas.clear_animations()

    dots_graphic.data[:, 0] = (x[frame_tracker] + x_init).astype(np.float32)

fig.add_animations(update_springs)
fig.show()

RFBOutputContext()

In [22]:
X = np.column_stack((x, v))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

In [23]:
ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

In [24]:
r_2 = r2_score(y_test, y_pred_ridge)
mse = mean_squared_error(y_test, y_pred_ridge)

print(f"{r_2:.4f}", f"{mse:.4f}")

1.0000 0.0000


In [30]:
print("Learned Weights (Coefficients):")
print(ridge_model.coef_)
print("Learned Bias (Intercept):")
print(ridge_model.intercept_)

Learned Weights (Coefficients):
[0.28447068 0.16490317 0.10775044 0.14921271 0.16444514 0.1701979
 0.17014679 0.16892327 0.15973839 0.15440471 0.24471118 0.4826077
 0.3698955  0.27519609 0.21008613 0.18548096 0.199911   0.30574517
 0.36278232 0.39240754]
Learned Bias (Intercept):
-0.9461944014764312


In [26]:
weights = ridge_model.coef_
labels = [
    f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
]

fig = go.Figure(
    data=[
        go.Bar(
            x=labels,
            y=weights,
            marker_color=np.where(
                weights >= 0, "royalblue", "firebrick"
            ),  # Blue for +, Red for -
        )
    ]
)

fig.update_layout(
    title="Reservoir Node Contribution (Feature Weights)",
    xaxis_title="Spring/Mass Node",
    yaxis_title="Weight Value",
    template="plotly_white",
)

fig.show()

# Spring Ring

In [3]:
time = 1000
dt = 0.005
steps = int(time / dt)
tau_steps = int(1.5 / dt)

test_size = 0.2

N = 10

m_diag = np.random.uniform(0.1, 0.5, size=N)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(np.random.uniform(0.1, 0.3, size=N))

k_vals = np.random.uniform(1, 8, N)
K = np.zeros((N, N))
idx = np.arange(N - 1)
K[idx, idx + 1] = -k_vals[:-1]
K[idx + 1, idx] = -k_vals[:-1]
K[np.arange(N), np.arange(N)] = k_vals + np.roll(k_vals, shift=1)
K[0, N - 1] = -k_vals[-1]
K[N - 1, 0] = -k_vals[-1]


t = np.linspace(0, time, steps)

u = np.sin(t)

In [4]:
@njit
def run_simulation(steps, dt, u, M_INV, K, C, N):
    x = np.zeros((steps, N))
    v = np.zeros((steps, N))

    acc = np.zeros(N)

    for i in range(1, steps):
        u_force = np.zeros(N)
        u_force[0] = u[i - 1]

        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + u_force)

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        u_force_next = np.zeros(N)
        u_force_next[0] = u[i]

        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + u_force_next)

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [5]:
x, v = run_simulation(steps, dt, u, M_INV, K, DAMP, N)

In [ ]:
radius = 5.0
thetas = np.linspace(0, 2 * np.pi, N, endpoint=False)

x_init = (radius * np.cos(thetas)).astype(np.float32)
y_init = (radius * np.sin(thetas)).astype(np.float32)
dots_data = np.column_stack([x_init, y_init])

fig = fpl.Figure()
dots_graphic = fig[0, 0].add_scatter(data=dots_data, sizes=15, colors="magenta")

frame_tracker = 0
frames_speed = 5
max_frames = 2000
def update_springs(canvas):
    global frame_tracker

    frame_tracker = (frame_tracker + frames_speed) % steps
    print(f"Frame: {frame_tracker}/{steps}", end="\r")

    if frame_tracker >= max_frames:
        canvas.clear_animations()

    # Linear to Angular Displ
    delta_theta = x[frame_tracker] / radius
    current_thetas = thetas + delta_theta

    dots_graphic.data[:, 0] = (radius * np.cos(current_thetas)).astype(np.float32)
    dots_graphic.data[:, 1] = (radius * np.sin(current_thetas)).astype(np.float32)


fig.add_animations(update_springs)
fig.show()

RFBOutputContext()

In [10]:
X = np.column_stack((x, v))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

In [11]:
ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

In [13]:
r_2 = r2_score(y_test, y_pred_ridge)
mse = mean_squared_error(y_test, y_pred_ridge)

print(f"{r_2:.4f}", f"{mse:.4f}")

1.0000 0.0000


In [14]:
print("Learned Weights (Coefficients):")
print(ridge_model.coef_)
print("Learned Bias (Intercept):")
print(ridge_model.intercept_)

Learned Weights (Coefficients):
[ 0.1086972   0.03281568 -0.00472747  0.03398241  0.05557676  0.07094292
  0.06126016  0.054194    0.04865808  0.06627046  0.06236501  0.04648088
  0.06992753  0.06473058  0.05477786  0.03665313  0.05165223  0.05925902
  0.05741199  0.04308069]
Learned Bias (Intercept):
-0.0008130901079976727


In [19]:
weights = ridge_model.coef_
labels = [
    f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
]

fig = go.Figure(
    data=[
        go.Bar(
            x=labels,
            y=weights,
            marker_color=np.where(
                weights >= 0, "royalblue", "firebrick"
            ),  # Blue for +, Red for -
        )
    ]
)

fig.update_layout(
    title="Reservoir Node Contribution (Feature Weights)",
    xaxis_title="Spring/Mass Node",
    yaxis_title="Weight Value",
    template="plotly_white",
)

fig.show()